# Airline Passenger Satisfaction Model

This notebook walks through training a supervised classifier that predicts whether an airline passenger reports being *satisfied* or *neutral/dissatisfied* based on survey responses from the Kaggle dataset.

## Workflow overview

We'll follow these steps:
1. Load and clean the survey data.
2. Prepare feature pipelines for numeric and categorical columns.
3. Train a balanced random forest classifier.
4. Evaluate the model on a stratified hold-out split.
5. Persist the trained pipeline and metrics for later reuse.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## Configuration

Define reusable configuration values such as paths, column lists, and label mappings.

In [ ]:
RANDOM_STATE = 42
TARGET_COLUMN = "satisfaction"
DROP_COLUMNS = ["Unnamed: 0", "id"]# "Unnamed: 0" and "id" are common artifacts: the former often appears when a CSV file has its index saved as a column, and the latter is typically just a unique identifier that carries no predictive value. Dropping them helps ensure models learn from meaningful features rather than redundant or non-informative columns
CATEGORICAL_MAPPINGS = {
    TARGET_COLUMN: {"satisfied": 1, "neutral or dissatisfied": 0},
}#CATEGORICAL_MAPPINGS is dict[str, dict[str, int]] transforms the target column into a binary representation suitable for scikit-learn estimators that expect numeric targets.

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "datasets").exists():#if this notebook is not run from the project root, move up one level
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "datasets/train.csv"
MODEL_PATH = PROJECT_ROOT / "models/satisfaction_model.joblib"
METRICS_PATH = PROJECT_ROOT / "reports/metrics.json"
TEST_SIZE = 0.2

## Helper functions

In [ ]:
def load_data(csv_path: Path) -> pd.DataFrame:
    """Load dataset and perform initial cleaning."""
    df = pd.read_csv(csv_path)

    for column in DROP_COLUMNS:
        if column in df.columns:
            df = df.drop(columns=column)
    # Map target labels to numeric values. Calling .map(mapping_dict) walks through that Series value-by-value and looks up each entry in the dictionary you provide. When it finds a match, it replaces the original string with the associated numeric value; if it can’t find a key, it returns NaN.
    df[TARGET_COLUMN] = df[TARGET_COLUMN].map(CATEGORICAL_MAPPINGS[TARGET_COLUMN])

    if df[TARGET_COLUMN].isna().any():#checks if prior mapping generated any NaN value, in that case throws an exception with a message and since this exception is not handled execution stops
        missing_targets = df[df[TARGET_COLUMN].isna()].shape[0]
        raise ValueError(
            f"Target column '{TARGET_COLUMN}' contains {missing_targets} unmapped values."
        )

    return df


def build_pipeline(categorical_features: list[str], numeric_features: list[str]) -> Pipeline:
    """Return a preprocessing + model pipeline."""
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, numeric_features),
            ("categorical", categorical_transformer, categorical_features),
        ]
    )

    model = RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1,
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", model),
        ]
    )

    return pipeline


def evaluate_model(pipeline: Pipeline, X_test: pd.DataFrame, y_test: pd.Series) -> dict[str, object]:
    """Compute evaluation metrics for the fitted pipeline."""
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "classification_report": classification_report(y_test, y_pred, output_dict=True),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
    }

    return metrics

## Load and inspect the data

Load the CSV, drop unused columns, map target labels to numeric values, and preview the result.

In [28]:
df = load_data(DATA_PATH)

print(f"Shape: {df.shape}")
df.head()

Shape: (103904, 23)


,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,3,1,...,5,4,3,4,4,5,5,25,18.0,0
1,Male,disloyal Customer,25,Business travel,Business,235,3,2,3,3,...,1,1,5,3,1,4,1,1,6.0,0
2,Female,Loyal Customer,26,Business travel,Business,1142,2,2,2,2,...,5,4,3,4,4,4,5,0,0.0,1
3,Female,Loyal Customer,25,Business travel,Business,562,2,5,5,5,...,2,2,5,3,1,4,2,11,9.0,0
4,Male,Loyal Customer,61,Business travel,Business,214,3,3,3,3,...,3,3,4,4,3,3,3,0,0.0,1


## Separate features and target

Split the preprocessed frame into feature matrix `X` and target vector `y`, and record which columns need categorical vs numeric handling.

In [29]:
y = df[TARGET_COLUMN]
X = df.drop(columns=[TARGET_COLUMN])

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = [col for col in X.columns if col not in categorical_features]

print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print(f"Numeric features ({len(numeric_features)}): first five -> {numeric_features[:5]}")

Categorical features (4): ['Gender', 'Customer Type', 'Type of Travel', 'Class']
Numeric features (18): first five -> ['Age', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking']


## Build the preprocessing + model pipeline

Instantiate the column transformer and random forest classifier as a single pipeline so that training and inference share the exact same steps.

In [ ]:
pipeline = build_pipeline(categorical_features, numeric_features)#build_pipeline(list[str], list[str]) -> Pipeline. ensures every fit or prediction shares the exact imputation, scaling, and encoding configuration alongside the random forest classifier.
pipeline

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## Train/test split and model fitting

Create an 80/20 stratified split to preserve class balance and fit the pipeline on the training fold.

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## Evaluate the model

Compute standard classification metrics on the hold-out set and visualise the detailed report.

In [32]:
metrics = evaluate_model(pipeline, X_test, y_test)
summary = {key: round(value, 4) for key, value in metrics.items() if isinstance(value, float)}
print(json.dumps(summary, indent=2))

report_df = pd.DataFrame(metrics["classification_report"]).T
report_df

{
  "accuracy": 0.9635,
  "precision": 0.9697,
  "recall": 0.9454,
  "f1": 0.9574,
  "roc_auc": 0.9943
}


,precision,recall,f1-score,support
0,0.959007,0.977412,0.968122,11776.000000
1,0.969700,0.945364,0.957377,9005.000000
accuracy,0.963524,0.963524,0.963524,0.963524
macro avg,0.964354,0.961388,0.962750,20781.000000
weighted avg,0.963641,0.963524,0.963466,20781.000000


### Confusion matrix

Review the confusion matrix counts for each class.

In [33]:
conf_matrix = pd.DataFrame(
    metrics["confusion_matrix"],
    index=["Actual neutral/dissatisfied", "Actual satisfied"],
    columns=["Pred neutral/dissatisfied", "Pred satisfied"],
)
conf_matrix

,Pred neutral/dissatisfied,Pred satisfied
Actual neutral/dissatisfied,11510,266
Actual satisfied,492,8513


## Persist the trained artefacts

Save the fitted pipeline and metrics JSON so they can be reused for inference or reporting later on.

In [34]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(pipeline, MODEL_PATH)

serialisable_metrics = {
    key: (round(value, 4) if isinstance(value, float) else value)
    for key, value in metrics.items()
}

with METRICS_PATH.open("w", encoding="utf-8") as f:
    json.dump(serialisable_metrics, f, indent=2)

print(f"Model saved to: {MODEL_PATH}")
print(f"Metrics saved to: {METRICS_PATH}")

Model saved to: C:\Users\raquel\Desktop\F5ProjectVI_ProblemaDeClasificacion\models\satisfaction_model.joblib
Metrics saved to: C:\Users\raquel\Desktop\F5ProjectVI_ProblemaDeClasificacion\reports\metrics.json
